# Condition Comparison: Significance Analysis

Compare predictions across any axis (layers, models, RMSD thresholds, etc.) with paired bootstrap tests and Holm-Bonferroni correction.

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("HOME_PROJ_DIR", Path.cwd().resolve().parents[1]))
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

Working directory: /home/famo00001/kinodata-3D-affinity-prediction


In [2]:
from prob.paths_and_io import get_out_dir, get_exp_dirs
from prob.prob_stats import compare_two_conditions, compare_multiple_conditions
from prob.prob_config import get_ds_load_config

print("Imports successful!")

Imports successful!


In [3]:
# Setup configuration
config = get_ds_load_config(
    gnn_model_type="CGNN-3D",
    split_type="random-k-fold",
    filter_rmsd_max_value=2,
)

print(f"Output directory: {config.output_dir}")
print(f"Target directory: {config.target_dir}")

Output directory: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold
Target directory: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/targets


In [4]:
def load_predictions(output_dir, target_name, model_name, layer_num):
    """Load y_true and y_pred from saved predictions CSV."""
    exp_dirs = get_exp_dirs(output_dir, target=target_name, prob_model=model_name, layer_num=layer_num)
    pred_file = exp_dirs["artifacts"] / f"{model_name}_predictions.csv"
    
    if not pred_file.exists():
        raise FileNotFoundError(f"Predictions not found: {pred_file}")
    
    df = pd.read_csv(pred_file)
    return df["y_true"].to_numpy(), df["y_pred"].to_numpy()

print("✓ Helper function defined")

✓ Helper function defined


## Configuration

Set the target, model, and layers to compare:

In [7]:
# CONFIGURATION: Change these to compare different axes
TARGET_NAME = "hydrogen_bonds"
MODEL_NAME = "ridge"
LAYERS = [1, 2, 3]
RANDOM_STATE = 96

print(f"Target: {TARGET_NAME}")
print(f"Model: {MODEL_NAME}")
print(f"Layers to compare: {LAYERS}")

Target: hydrogen_bonds
Model: ridge
Layers to compare: [1, 2, 3]


## Load Predictions

In [8]:
# Load predictions for all layers
predictions = {}
for layer in LAYERS:
    try:
        y_true, y_pred = load_predictions(config.output_dir, TARGET_NAME, MODEL_NAME, layer)
        predictions[f"layer_{layer}"] = (y_true, y_pred)
        print(f"✓ Layer {layer}: {len(y_true)} samples")
    except FileNotFoundError as e:
        print(f"✗ Layer {layer}: {e}")

print(f"\nSuccessfully loaded {len(predictions)} layers")

✓ Layer 1: 4124 samples
✓ Layer 2: 4124 samples
✓ Layer 3: 4124 samples

Successfully loaded 3 layers


## Pairwise Comparison (Two Conditions)

Compare layer 1 vs layer 2 with detailed statistics:

In [9]:
if "layer_1" in predictions and "layer_2" in predictions:
    result = compare_two_conditions(
        "layer_1", predictions["layer_1"][0], predictions["layer_1"][1],
        "layer_2", predictions["layer_2"][0], predictions["layer_2"][1],
        random_state=RANDOM_STATE
    )
    
    print("\n" + "="*70)
    print("LAYER 1 vs LAYER 2 COMPARISON")
    print("="*70)
    print(f"\nSample size: {result['n_samples']}")
    print(f"\nR² Difference (Layer 1 - Layer 2):")
    print(f"  Point estimate:      {result['delta_r2']:+.4f}")
    print(f"  95% CI:              [{result['delta_r2_ci_low']:+.4f}, {result['delta_r2_ci_high']:+.4f}]")
    print(f"  Bootstrap p-value:   {result['delta_r2_bootstrap_pval']:.4f}")
    
    print(f"\nRMSE Difference (Layer 1 - Layer 2):")
    print(f"  Point estimate:      {result['delta_rmse']:+.4f}")
    print(f"  95% CI:              [{result['delta_rmse_ci_low']:+.4f}, {result['delta_rmse_ci_high']:+.4f}]")
    print(f"  Bootstrap p-value:   {result['delta_rmse_bootstrap_pval']:.4f}")
    
    print(f"\nWilcoxon signed-rank test (squared errors):")
    print(f"  p-value:             {result['wilcoxon_pval']:.4f}")
    print(f"  Median Δ(squared err): {result['median_delta_squared_error']:+.4f}")
else:
    print("Not enough layers loaded to compare")


LAYER 1 vs LAYER 2 COMPARISON

Sample size: 4124

R² Difference (Layer 1 - Layer 2):
  Point estimate:      -0.0140
  95% CI:              [-0.0346, +0.0067]
  Bootstrap p-value:   0.2000

RMSE Difference (Layer 1 - Layer 2):
  Point estimate:      +0.0104
  95% CI:              [-0.0049, +0.0257]
  Bootstrap p-value:   0.2000

Wilcoxon signed-rank test (squared errors):
  p-value:             0.0443
  Median Δ(squared err): +0.0130


## Multiple Comparisons (All Pairs)

Compare all layers at once with Holm-Bonferroni correction:

In [10]:
if len(predictions) >= 2:
    comparison_df = compare_multiple_conditions(predictions, random_state=RANDOM_STATE)
    
    print(f"\nAll pairwise comparisons ({len(comparison_df)} pairs):")
    print("="*100)
    
    # Display key columns
    display_cols = [
        "condition_a", "condition_b",
        "delta_r2", "delta_r2_bootstrap_pval", "delta_r2_bootstrap_pval_holm",
        "delta_rmse", "wilcoxon_pval", "wilcoxon_pval_holm"
    ]
    print(comparison_df[display_cols].to_string(index=False))
else:
    print("Need at least 2 conditions to compare")


All pairwise comparisons (3 pairs):
condition_a condition_b  delta_r2  delta_r2_bootstrap_pval  delta_r2_bootstrap_pval_holm  delta_rmse  wilcoxon_pval  wilcoxon_pval_holm
    layer_1     layer_2 -0.013956                    0.200                         0.600    0.010431       0.044267            0.132800
    layer_1     layer_3 -0.006592                    0.628                         0.932    0.004917       0.245767            0.491535
    layer_2     layer_3  0.007364                    0.466                         0.932   -0.005514       0.519914            0.519914


## Summary with Significance Markers

In [11]:
if len(predictions) >= 2:
    print("\nSummary of Holm-Bonferroni corrected results:")
    print("-" * 80)
    
    for idx, row in comparison_df.iterrows():
        r2_sig = "***" if row["delta_r2_bootstrap_pval_holm"] < 0.001 else "**" if row["delta_r2_bootstrap_pval_holm"] < 0.01 else "*" if row["delta_r2_bootstrap_pval_holm"] < 0.05 else "ns"
        rmse_sig = "***" if row["delta_rmse_bootstrap_pval_holm"] < 0.001 else "**" if row["delta_rmse_bootstrap_pval_holm"] < 0.01 else "*" if row["delta_rmse_bootstrap_pval_holm"] < 0.05 else "ns"
        wilcox_sig = "***" if row["wilcoxon_pval_holm"] < 0.001 else "**" if row["wilcoxon_pval_holm"] < 0.01 else "*" if row["wilcoxon_pval_holm"] < 0.05 else "ns"
        
        print(f"\n{row['condition_a']} vs {row['condition_b']}:")
        print(f"  ΔR² = {row['delta_r2']:+.4f} [{row['delta_r2_ci_low']:+.4f}, {row['delta_r2_ci_high']:+.4f}] {r2_sig}")
        print(f"  ΔRMSE = {row['delta_rmse']:+.4f} [{row['delta_rmse_ci_low']:+.4f}, {row['delta_rmse_ci_high']:+.4f}] {rmse_sig}")
        print(f"  Wilcoxon p_holm = {row['wilcoxon_pval_holm']:.4f} {wilcox_sig}")
    
    print("\nSignificance codes: *** p<0.001  ** p<0.01  * p<0.05  ns=not significant")
else:
    print("Need at least 2 conditions to compare")


Summary of Holm-Bonferroni corrected results:
--------------------------------------------------------------------------------

layer_1 vs layer_2:
  ΔR² = -0.0140 [-0.0346, +0.0067] ns
  ΔRMSE = +0.0104 [-0.0049, +0.0257] ns
  Wilcoxon p_holm = 0.1328 ns

layer_1 vs layer_3:
  ΔR² = -0.0066 [-0.0321, +0.0175] ns
  ΔRMSE = +0.0049 [-0.0131, +0.0237] ns
  Wilcoxon p_holm = 0.4915 ns

layer_2 vs layer_3:
  ΔR² = +0.0074 [-0.0157, +0.0297] ns
  ΔRMSE = -0.0055 [-0.0221, +0.0117] ns
  Wilcoxon p_holm = 0.5199 ns

Significance codes: *** p<0.001  ** p<0.01  * p<0.05  ns=not significant


## Save Results

In [13]:
if len(predictions) >= 2:
    output_file = Path(config.output_dir) / TARGET_NAME / MODEL_NAME / f"{MODEL_NAME}_layer_comparison.csv"
    output_file.parent.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(output_file, index=False)
    
    print(f"✓ Results saved to: {output_file}")
    print(f"\nDataFrame shape: {comparison_df.shape}")
    print(f"Columns: {', '.join(comparison_df.columns.tolist())}")
else:
    print("Cannot save: need at least 2 conditions")

✓ Results saved to: /home/famo00001/kinodata-3D-affinity-prediction/data/probing/CGNN-3D/rmsd_cutoff_2/random-k-fold/hydrogen_bonds/ridge/ridge_layer_comparison.csv

DataFrame shape: (3, 16)
Columns: condition_a, condition_b, n_samples, delta_r2, delta_r2_ci_low, delta_r2_ci_high, delta_r2_bootstrap_pval, delta_rmse, delta_rmse_ci_low, delta_rmse_ci_high, delta_rmse_bootstrap_pval, median_delta_squared_error, wilcoxon_pval, delta_r2_bootstrap_pval_holm, delta_rmse_bootstrap_pval_holm, wilcoxon_pval_holm
